# Causal Inference I: Correlation, Causation, and Counterfactuals


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/14-Causal-Inference-I.ipynb)

So far in this course we have built tools that are very good at answering one kind of
question: **given what I observe, what should I predict?** Clustering, regression, and
classification all learn associations in data.

But many of the questions we actually care about as data scientists are of a different
kind. They are questions about **what would happen if we acted**:

* If we *change* the checkout button from blue to green, will more people buy?
* If a patient *takes* this drug, will they recover faster?
* If a student *enrolls* in the tutoring program, will their grade go up?
* If the city *raises* the minimum wage, will employment fall?


These are **causal** questions, and — as we will see — no amount of predictive accuracy
answers them by itself.



# A fitted regression is not a causal claim

## You just fit regressions. What does a coefficient mean?

In Lectures 10 and 11 you fit models like

```python
smf.ols("Lottery ~ Literacy + Wealth + Region", data=df).fit().summary()
```

and read off a coefficient, a confidence interval, and a $p$-value for each variable.

There are **two very different readings** of a coefficient $\hat\beta_1$ on $X$:

**Associational reading (what OLS estimates)**

"Among the units in my data, those with $X$ one unit higher have $Y$ higher by
$\hat\beta_1$ on average."

**Causal reading (what a decision-maker wants)**

"If I *raised* $X$ by one unit, $Y$ would *change* by $\hat\beta_1$ on average."


Least squares gives you the first reading, always. It gives you the second **only under
extra assumptions that the data cannot check for you.** This lecture is about what those
assumptions are, and what happens when they fail.


## A regression you would recognize

Two variables measured daily across the summer: **ice cream sales** and **drowning deaths**.
Let's fit the regression exactly as we did in Lectures 10 and 11.

In [ ]:
#| echo: true
import numpy as np, pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(701)
n = 200
temp = rng.uniform(60, 100, n)                        # daily high temperature (°F)
ice_cream = 2.0 * temp + rng.normal(0, 15, n)         # ice cream sales
drownings = 0.10 * temp + rng.normal(0, 2.0, n)       # drowning deaths
# NOTE: ice cream does NOT appear in the equation for drownings.

df = pd.DataFrame({"ice_cream": ice_cream, "drownings": drownings, "temp": temp})
res = smf.ols("drownings ~ ice_cream", data=df).fit()
print(res.summary().tables[1])

A positive slope, a tight confidence interval, $p \approx 0$. Every diagnostic from
Lecture 10 says the fit is fine.
**Should the city ban ice cream to reduce drownings?**


## Correlation is not causation

* The regression is *correct* as a description of the data: on days with more ice cream
  sales there **are** more drownings.
* But look at how the data were generated: a third variable, **hot weather**, drives both.
  Hot days increase ice cream sales *and* send more people swimming.
* Temperature is a **confounder**. Ice cream has **no** effect on drowning, and banning it
  would change nothing.


The slogan "correlation is not causation" is famous, and yet the mistakes it warns about
are everywhere — most often hidden inside a regression table that *looks* authoritative.


## The picture behind the slogan

In [ ]:
#| echo: false
#| fig-align: center
import networkx as nx
import matplotlib.pyplot as plt

def draw_fork(ax, z, x, y, title):
    """Common cause z -> x, z -> y (Lecture 15's drawing style), plus the
    association we *see* between x and y, dashed and crossed out."""
    pos = {z: (1, 1), x: (0, 0), y: (2, 0)}
    G = nx.DiGraph([(z, x), (z, y)])
    nx.draw(G, pos, ax=ax, with_labels=True, node_color="#cde", node_size=3200,
            font_size=10, arrowsize=22, edgecolors="#333", linewidths=1.2)
    nx.draw_networkx_edges(nx.DiGraph([(x, y)]), pos, ax=ax, node_size=3200,
                           style="dashed", edge_color="#c0392b", width=2, arrowsize=22)
    ax.text(1, 0, "✗", color="#c0392b", fontsize=22, ha="center", va="center",
            bbox=dict(boxstyle="circle,pad=0.1", fc="white", ec="none"))
    ax.text(1, -0.22, "association, not cause", color="#c0392b",
            fontsize=9, ha="center", va="top")
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-0.5, 1.4)
    ax.set_title(title, fontsize=11)

fig, ax = plt.subplots(figsize=(5.5, 3.4))
draw_fork(ax, "Temperature", "Ice cream\nsales", "Drownings",
          "Temperature is a common cause of both")
plt.tight_layout(); plt.show()

Both arrows point *out of* temperature. Follow them and you can get from ice cream to
drownings without either one causing the other. In Lecture II this shape gets a name and a
set of rules; for now, the shape is the whole story.


## Confounding, visualized

In [ ]:
#| echo: false
#| fig-align: center
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4), sharey=True, constrained_layout=True)
cmap = plt.get_cmap("coolwarm")

sc = ax[0].scatter(ice_cream, drownings, c=temp, cmap=cmap, s=25)
b_all = np.polyfit(ice_cream, drownings, 1)
xs = np.array([ice_cream.min(), ice_cream.max()])
ax[0].plot(xs, np.polyval(b_all, xs), c="k", lw=2, label=f"OLS slope {b_all[0]:+.3f}")
ax[0].set_xlabel("Ice cream sales")
ax[0].set_ylabel("Drownings")
ax[0].set_title("All days: strong positive association")
ax[0].legend(loc="upper left")

# Hold temperature (nearly) fixed: fit a separate line inside each 10°F band
ax[1].plot(xs, np.polyval(b_all, xs), c="k", lw=2, ls="--", label="pooled line (left)")
for lo in (60, 70, 80, 90):
    band = (temp >= lo) & (temp < lo + 10)
    col = cmap((lo + 5 - temp.min()) / (temp.max() - temp.min()))
    b = np.polyfit(ice_cream[band], drownings[band], 1)
    xs_b = np.array([ice_cream[band].min(), ice_cream[band].max()])
    ax[1].scatter(ice_cream[band], drownings[band], color=col, s=25, alpha=0.6)
    ax[1].plot(xs_b, np.polyval(b, xs_b), color=col, lw=2.5,
               path_effects=None, label=f"{lo}–{lo+10}°F: slope {b[0]:+.3f}")
ax[1].set_xlabel("Ice cream sales")
ax[1].set_title("Within each 10°F band: no consistent slope")
ax[1].legend(loc="upper left", fontsize=8)

fig.colorbar(sc, ax=ax, label="Temperature (°F)", shrink=0.9)
plt.show()

**Hold the confounder fixed** and the relationship falls apart: within any one temperature
band the fitted slope is small, not significant, and flips sign from band to band. The
steep pooled line comes entirely from moving *between* bands — that is, from temperature.
There was never a causal link.


## Confounding

**Definition**

A **confounder** is a variable that causally affects **both** the treatment (or explanatory
variable) $X$ **and** the outcome $Y$. It creates an association between $X$ and $Y$ that
is not due to $X$ causing $Y$.


Multiple regression can *sometimes* fix this — **if** we measure the confounder and include it:

In [ ]:
#| echo: true
naive = smf.ols("drownings ~ ice_cream", data=df).fit().params["ice_cream"]
adj   = smf.ols("drownings ~ ice_cream + temp", data=df).fit().params["ice_cream"]
print(f"slope on ice_cream, ignoring temp:  {naive:+.3f}")
print(f"slope on ice_cream, adjusting temp: {adj:+.3f}   (truth: 0)")

Adding `temp` to the formula drives the ice-cream coefficient to zero. So "controlling for"
a variable is just adding it to the regression — but **which** variables to add is a causal
question that the fitting procedure cannot answer. If temperature had not been measured, no
amount of model selection would have saved us.


## Same trap, higher stakes

A hospital analytics team fits a model on patient records and finds a strong,
statistically significant relationship:

> Patients who **visit the hospital** are *more* likely to die within the year than
> patients who do not.

Should we conclude that hospitals *cause* death and recommend people avoid them?


Of course not. Sick people go to hospitals. The hospital visit is **associated** with
death because both are driven by a common cause — being sick. Same shape as the ice cream
picture, with the names changed.

In [ ]:
#| echo: false
#| fig-align: center
fig, ax = plt.subplots(figsize=(4.4, 2.7))
draw_fork(ax, "Sickness", "Hospital\nvisit", "Death",
          "Sickness is a common cause of both")
plt.tight_layout(); plt.show()

The model is a perfectly good *predictor* ("I see a hospital visit, I predict higher
mortality") and a catastrophically bad *guide to action* ("close the hospitals"). That
gap — **prediction** vs. **intervention** — is the subject of these two lectures
[@angrist2009mostly].


# Prediction vs. intervention

## Two different questions

It is worth being precise about the distinction.


**Prediction (association)**

$$ P(Y \mid X = x) $$

"Among people I *observe* to have $X = x$, what is the distribution of $Y$?"

This is what supervised learning — including every regression you have fit — estimates.

**Intervention (causation)**

$$ P(Y \mid do(X = x)) $$

"If I *set* $X = x$ for everyone, what is the distribution of $Y$?"

This is what a decision-maker needs. [@pearl2009causality]


In general $P(Y \mid X=x) \neq P(Y \mid do(X=x))$. The whole game is figuring out when —
and how — we can recover the second from data that only shows us the first.


## The ladder of causation

Judea Pearl describes three "rungs" of reasoning [@pearl2018why]:

| Rung | Question | Example |
|------|----------|---------|
| 1. **Association** | What is? | *Seeing* — customers who buy diapers also buy beer |
| 2. **Intervention** | What if I do? | *Doing* — if I discount diapers, will beer sales rise? |
| 3. **Counterfactual** | What if I had? | *Imagining* — would this patient have recovered had they not taken the drug? |

: {tbl-colwidths="[18,22,60]"}

Standard machine learning lives almost entirely on **Rung 1**. Causal inference is about
climbing to Rungs 2 and 3.


## Where do spurious associations come from?

A correlation between $X$ and $Y$ can arise for several reasons — only one of which is
"$X$ causes $Y$":

* **$X$ causes $Y$** (the thing we want),
* **$Y$ causes $X$** (reverse causation),
* **a confounder $Z$ causes both** $X$ and $Y$ (ice cream / drowning),
* **collider bias** — conditioning or selecting on a *common effect* of $X$ and $Y$
  (Lecture II),
* **pure chance** — with enough variables, some will correlate by luck.[^spurious]


[^spurious]: See Tyler Vigen's *Spurious Correlations* (<https://tylervigen.com/spurious-correlations>) for entertaining examples, e.g. US cheese consumption vs. deaths by bedsheet entanglement.

# Potential outcomes

## The potential outcomes framework

To reason precisely, we use the **Neyman–Rubin potential outcomes** model
[@splawa1990application; @rubin1974estimating]. Our running example: does an **optional
tutoring program** raise students' **exam scores**? Students choose whether to sign up.

For each unit $i$ (here, a student) and a binary **treatment** $T_i \in \{0,1\}$ (tutored
or not):

* $Y_i(1)$ — the exam score we *would* see if student $i$ were tutored,
* $Y_i(0)$ — the exam score we *would* see if student $i$ were **not** tutored.


The **individual treatment effect** is
$$ \tau_i = Y_i(1) - Y_i(0). $$


These are called *potential* outcomes because, for any given unit, only **one** of them
is ever realized.


## If we could see everything (we can't)

Imagine an all-seeing observer who knows **both** potential outcomes for six students.
The three weaker students signed up for tutoring; the three stronger ones did not.

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | 60 | 62 | 2 |
| Bob  | 1 | 65 | 67 | 2 |
| Cara | 1 | 70 | 72 | 2 |
| Dan  | 0 | 68 | 70 | 2 |
| Eve  | 0 | 70 | 72 | 2 |
| Finn | 0 | 72 | 74 | 2 |

* Tutoring raises **every** student's score by exactly 2 points, so the average effect
  is 2.
* Yet compare the scores we would *actually observe*: tutored students average
  $(62 + 67 + 72)/3 = 67$; untutored students average $(68 + 70 + 72)/3 = 70$.
  The naive comparison says tutoring **lowers** scores by 3.
* The gap is in the $Y_i(0)$ column: those who chose tutoring would have scored $65$ on
  average without it, versus $70$ for those who didn't. That $-5$ is **not** an effect of
  tutoring — it is who signed up.


## The fundamental problem of causal inference

Nobody is all-seeing. We only ever observe the outcome for the treatment actually received:

$$ Y_i^{\text{obs}} = T_i\, Y_i(1) + (1 - T_i)\, Y_i(0). $$

| Student | $T_i$ | $Y_i(0)$ | $Y_i(1)$ | $\tau_i$ |
|:-------:|:-----:|:--------:|:--------:|:--------:|
| Ann  | 1 | [?]{style="color:#b0b0b0"} | 62 | [?]{style="color:#b0b0b0"} |
| Bob  | 1 | [?]{style="color:#b0b0b0"} | 67 | [?]{style="color:#b0b0b0"} |
| Cara | 1 | [?]{style="color:#b0b0b0"} | 72 | [?]{style="color:#b0b0b0"} |
| Dan  | 0 | 68 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Eve  | 0 | 70 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |
| Finn | 0 | 72 | [?]{style="color:#b0b0b0"} | [?]{style="color:#b0b0b0"} |

The same table, as the world shows it to us. The other potential outcome — the
**counterfactual** — is missing, and every $\tau_i$ needs a grey cell.


We can never compute $\tau_i$ for a single individual. Causal inference is fundamentally
a **missing data problem** [@imbens2015causal]. (The equation also quietly assumes each
student's score depends only on their *own* treatment — Lecture II names that assumption.)


## The Average Treatment Effect (ATE)

Since individual effects are unknowable, we target **population averages**.

**Definition**

The **Average Treatment Effect** is the expected difference between a unit's two potential
outcomes, averaged over the whole population:
$$ \text{ATE} = \mathbb{E}[\,Y(1) - Y(0)\,] = \mathbb{E}[Y(1)] - \mathbb{E}[Y(0)]. $$


On the tutoring table, $\text{ATE} = 2$. Related quantities you will meet:

- **ATT** — the effect *on the treated*, $\mathbb{E}[Y(1) - Y(0)\mid T=1]$. Answers
  "did the program help the people we actually enrolled?" — the question a program
  manager is asked.
- **CATE** — the *conditional* effect for a subgroup, $\mathbb{E}[Y(1) - Y(0)\mid X=x]$.
  Answers "for *whom* does it work?" — the basis of heterogeneous treatment effects and
  uplift modeling.


## Why we can't just compare treated vs. untreated

The tempting estimator is the **naive difference in means**:

$$ \underbrace{\mathbb{E}[Y \mid T=1] - \mathbb{E}[Y \mid T=0]}_{\text{what we can measure}}. $$

Add and subtract the missing counterfactual $\mathbb{E}[Y(0)\mid T=1]$ to decompose it:

$$
\underbrace{\mathbb{E}[Y(1)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{naive difference}}
= \underbrace{\text{ATT}}_{\text{causal}}
+ \underbrace{\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]}_{\text{selection bias}}.
$$


Check it against the all-seeing table: $\text{ATT} = 2$, selection bias $= 65 - 70 = -5$,
and $2 + (-5) = -3$ — exactly the naive difference. The arithmetic was right; the causal
reading was wrong.


## Selection bias

**Definition**

**Selection bias** is the difference in outcomes that treated and untreated units would
have shown *even without any treatment*: $\mathbb{E}[Y(0)\mid T{=}1] - \mathbb{E}[Y(0)\mid T{=}0]$.
It is nonzero whenever *who gets treated* depends on things that also affect the outcome.


* Exactly the hospital example: patients who visit are sicker to begin with. And exactly
  the tutoring table: the students who signed up were weaker to begin with.
* The coefficient on `T` in the regression `Y ~ T` *is* the naive difference — so it
  inherits the same bias. Our job is to make that term vanish.


**A note on terminology**

We use *selection bias* in the sense above — a baseline difference between those who were
and were not treated (the econometrics usage). Epidemiologists reserve the term for bias
caused by how units get *into the sample*. Lecture II calls that **collider bias** and
treats it as a separate mechanism; keep the two apart.


# Simpson's paradox

## Confounding in numbers: a kidney stone study

A famous real study compared two treatments for kidney stones
[@charig1986comparison]. Here are the recovery rates:

| | Treatment A (surgery) | Treatment B (less invasive) |
|:--|:--:|:--:|
| **Small stones** | 81/87 = **93%** | 234/270 = **87%** |
| **Large stones** | 192/263 = **73%** | 55/80 = **69%** |
| **Overall** | 273/350 = **78%** | 289/350 = **83%** |

Treatment A wins for **small** stones **and** for **large** stones — yet Treatment B wins
**overall**.


## What is going on?

In [ ]:
#| echo: true
# Recovery counts (successes, total) by treatment and stone size
data = {
    "A": {"small": (81, 87),  "large": (192, 263)},
    "B": {"small": (234, 270), "large": (55, 80)},
}

for tx in ("A", "B"):
    s_succ, s_tot = data[tx]["small"]
    l_succ, l_tot = data[tx]["large"]
    overall = (s_succ + l_succ) / (s_tot + l_tot)
    print(f"Treatment {tx}: small={s_succ/s_tot:.0%}, "
          f"large={l_succ/l_tot:.0%}, overall={overall:.0%}")

**Stone size is a confounder.** Doctors gave the risky Treatment A to the *hard* (large-stone)
cases and Treatment B to the *easy* (small-stone) cases. The overall numbers mix effect
with case-mix.


## The two fingerprints of a confounder

In [ ]:
#| echo: false
#| fig-align: center
import matplotlib.pyplot as plt

colors = {"A": "#2c7fb8", "B": "#d95f02"}
strata = ["small", "large"]
rate = {tx: {z: data[tx][z][0] / data[tx][z][1] for z in strata} for tx in "AB"}
n    = {tx: {z: data[tx][z][1] for z in strata} for tx in "AB"}
for tx in "AB":
    n[tx]["overall"] = sum(data[tx][z][1] for z in strata)
    rate[tx]["overall"] = sum(data[tx][z][0] for z in strata) / n[tx]["overall"]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 3.4),
                               gridspec_kw={"width_ratios": [2.2, 1]})

# Left: recovery rate by stratum, bar width proportional to number of patients
groups = ["small", "large", "overall"]
gap = 0.03
for i, z in enumerate(groups):
    for tx, side in (("A", -1), ("B", +1)):
        w = 0.42 * n[tx][z] / 350
        left = i - gap - w if side < 0 else i + gap
        ax0.bar(left, rate[tx][z], width=w, align="edge", color=colors[tx],
                label=f"Treatment {tx}" if i == 0 else None)
        ax0.text(left + w / 2, rate[tx][z] + 0.02, f"{rate[tx][z]:.0%}",
                 ha="center", va="bottom", fontsize=9)
        ax0.text(left + w / 2, 0.04, f"n={n[tx][z]}", ha="center", va="bottom",
                 fontsize=8, color="white", rotation=90 if w < 0.2 else 0)
ax0.set_xticks(range(3), ["Small stones", "Large stones", "Overall"])
ax0.set_ylim(0, 1.25)
ax0.set_ylabel("Recovery rate")
ax0.set_title("Recovery rate (bar width ∝ number of patients)")
ax0.legend(loc="upper center", ncol=2, fontsize=9, frameon=False)

# Right: case mix of each treatment arm (100% stacked)
share = {tx: {z: n[tx][z] / n[tx]["overall"] for z in strata} for tx in "AB"}
for j, tx in enumerate("AB"):
    ax1.bar(j, share[tx]["small"], color="#bbbbbb", label="small stones" if j == 0 else None)
    ax1.bar(j, share[tx]["large"], bottom=share[tx]["small"], color="#555555",
            label="large stones" if j == 0 else None)
    ax1.text(j, share[tx]["small"] / 2, f"{share[tx]['small']:.0%} small",
             ha="center", va="center", fontsize=9)
    ax1.text(j, share[tx]["small"] + share[tx]["large"] / 2, f"{share[tx]['large']:.0%} large",
             ha="center", va="center", fontsize=9, color="white")
ax1.set_xticks([0, 1], ["Treatment A", "Treatment B"])
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
ax1.set_title("Case mix of each treatment arm")

plt.tight_layout()
plt.show()

Every Simpson's table carries **two fingerprints** of the confounder, both visible here:

1. **The case mix differs between the arms** — 75% of A's patients had large stones vs.
   23% of B's. Those two numbers are the ones to hunt for in any such table; they are why
   A's overall rate is dragged toward its hard cases and B's toward its easy ones.
2. **The strata differ in outcome** — large stones recover less often under *either*
   treatment.


## Give both arms the same case mix

The overall row compared A *on mostly large stones* with B *on mostly small stones*. The fair
question: **what would each treatment's recovery rate be if it had been given to the whole
study population** — the 51% small / 49% large mix we actually have?

In [ ]:
#| echo: true
n_small = data["A"]["small"][1] + data["B"]["small"][1]      # 357 patients
n_large = data["A"]["large"][1] + data["B"]["large"][1]      # 343 patients
p_small, p_large = n_small / (n_small + n_large), n_large / (n_small + n_large)
print(f"P(small) = {p_small:.2f}, P(large) = {p_large:.2f}")

for tx in ("A", "B"):
    rate = {z: data[tx][z][0] / data[tx][z][1] for z in ("small", "large")}
    adjusted = rate["small"] * p_small + rate["large"] * p_large
    print(f"Treatment {tx}: adjusted recovery = {adjusted:.0%}")

Adjusted, **A ≈ 83% beats B ≈ 78%** — the crude 78% vs. 83% flips back to agree with the
strata. This is the **adjustment formula**, also called **direct standardization**:

$$ P(Y \mid do(T{=}t)) \;=\; \sum_z P(Y \mid T{=}t,\, Z{=}z)\; P(Z{=}z), $$

each arm's within-stratum rates weighted by the **population's** stratum shares, not the
arm's own. The regression `Y ~ T + Z` is the same move in model form — compare within
strata, then average — which is why adding `temp` fixed the ice cream slope.


## Simpson's paradox

**Definition**

**Simpson's paradox** occurs when an association between $X$ and $Y$ that holds in
**every subgroup** of the data **reverses** (or disappears) when the subgroups are
combined. It happens when a third variable $Z$ is distributed differently across the
levels of $X$ *and* is itself related to $Y$ — the two fingerprints.


* The **within-stratum** comparisons hold $Z$ fixed. When $Z$ is a confounder — as stone
  size is here — they are the ones to trust, and standardizing just re-weights them.
  **Treatment A is better.**
* The **overall** comparison is contaminated by selection bias: A's patients were sicker
  to begin with.


The paradox is not a mathematical curiosity — it is a warning: **aggregated associations
can point the opposite direction from the truth.** But "always adjust" is not the lesson
either. Deciding *which* variables to adjust for is itself a causal question (Lecture II),
as the next example shows.


## Berkeley admissions, 1973

A second real example. UC Berkeley's graduate admissions data showed [@bickel1975sex]:

* **Overall:** men were admitted at a noticeably *higher* rate than women — suggesting bias.
* **Within most departments** (four of the six largest): women were admitted at an equal
  or slightly *higher* rate.


The third variable was **department**: women applied disproportionately to departments with
low admission rates. Same arithmetic as the kidney stones — but is department a
*confounder*? Department does not cause an applicant's gender. Rather, gender shapes which
department one applies to, which in turn shapes admission:
$$ \text{Gender} \rightarrow \text{Department} \rightarrow \text{Admission}. $$


Department sits *on the path* from gender to admission, so whether to adjust for it depends
on the question ("is the committee biased?" vs. "does the system admit fewer women?"). The
answer depends on the **arrows**, not on the table — exactly where Lecture II begins.


# Randomization

## The gold standard: randomized experiments

How do we break confounding by design? **Randomly assign** the treatment.

If a coin flip decides $T_i$, then treatment is *statistically independent* of the
potential outcomes:
$$ \{Y_i(0), Y_i(1)\} \perp\!\!\!\perp T_i. $$


Independence kills the selection-bias term, because the treated and untreated groups are —
in expectation — identical in every respect (measured *and* unmeasured) except the treatment:
$$ \mathbb{E}[Y(0)\mid T{=}1] = \mathbb{E}[Y(0)\mid T{=}0]. $$


Independence also gives $\mathbb{E}[Y(1)\mid T{=}1] = \mathbb{E}[Y(1)]$ and
$\mathbb{E}[Y(0)\mid T{=}1] = \mathbb{E}[Y(0)]$: the treated are a representative sample of
everyone, so the effect *on the treated* is the effect on everyone — $\text{ATT} = \text{ATE}$.


So the naive difference in means becomes an **unbiased estimate of the ATE**
[@imbens2015causal].


## Randomize the tutoring table

"Unbiased" means: averaged over all the ways the coin could have landed, the naive
difference hits the truth. With six students and three tutoring slots there are only
$\binom{6}{3} = 20$ possible assignments — so we can check every one.

In [ ]:
#| echo: true
from itertools import combinations

Y0 = np.array([60, 65, 70, 68, 70, 72])        # the all-seeing table's Y(0) column
Y1 = Y0 + 2                                    # tau_i = 2 for everyone
students = np.arange(6)

naive = []
for tutored in combinations(students, 3):      # every way to tutor exactly 3 of 6
    T = np.isin(students, tutored)
    naive.append(Y1[T].mean() - Y0[~T].mean())

print(f"{len(naive)} assignments; naive difference ranges "
      f"{min(naive):+.1f} to {max(naive):+.1f}; mean = {np.mean(naive):+.2f}")

Any single randomized study can miss (one assignment gives $-4.3$, another $+8.3$), but the
*average* over assignments is exactly the ATE of $2$. Self-selection, by contrast, picked
out one particular assignment — the weak students — and gave $-3$ every time.


## Randomization removes confounding

In [ ]:
#| echo: false
#| fig-align: center
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 2000
severity = rng.uniform(0, 1, n)          # a confounder: how sick each patient is

# TRUE treatment effect is +5 for everyone
def outcomes(T, severity):
    return 50 + 5 * T - 30 * severity + rng.normal(0, 3, len(T))

# Observational: sicker patients are MORE likely to get the drug
p_treat_obs = severity
T_obs = (rng.uniform(0, 1, n) < p_treat_obs).astype(int)
Y_obs = outcomes(T_obs, severity)
naive_obs = Y_obs[T_obs == 1].mean() - Y_obs[T_obs == 0].mean()

# Randomized: coin flip, independent of severity
T_rct = (rng.uniform(0, 1, n) < 0.5).astype(int)
Y_rct = outcomes(T_rct, severity)
naive_rct = Y_rct[T_rct == 1].mean() - Y_rct[T_rct == 0].mean()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.3, 1, 1]})

vals = [5, naive_obs, naive_rct]
axes[0].bar(["True effect", "Observational\n(confounded)", "Randomized\n(RCT)"], vals,
            color=["#2c7fb8", "#d95f02", "#1b9e77"])
axes[0].axhline(5, ls="--", c="gray")
axes[0].axhline(0, c="k", lw=0.8)
for i, v in enumerate(vals):
    axes[0].text(i, v + (0.3 if v >= 0 else -0.3), f"{v:+.1f}",
                 ha="center", va="bottom" if v >= 0 else "top")
axes[0].set_ylim(min(vals) - 1.5, max(vals) + 1.5)
axes[0].set_ylabel("Estimated treatment effect")
axes[0].set_title("Naive difference in means", fontsize=10)

bins = np.linspace(0, 1, 11)
for ax, T, col, title in ((axes[1], T_obs, "#d95f02", "Observational: who got the drug?"),
                          (axes[2], T_rct, "#1b9e77", "Randomized: who got the drug?")):
    ax.hist(severity[T == 0], bins, alpha=0.55, color="#888888", label="untreated")
    ax.hist(severity[T == 1], bins, alpha=0.55, color=col, label="treated")
    ax.set_xlabel("Severity (the confounder)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
axes[1].set_ylabel("Number of patients")

plt.tight_layout()
plt.show()

The true effect is $+5$. The **observational** estimate is badly biased, and the middle
panel shows the mechanism: the treated group is dominated by severe cases, who do worse for
reasons that have nothing to do with the drug. Under **randomization** the two severity
histograms lie on top of each other, and the naive estimate recovers the truth.

Try it yourself in the notebook: change `p_treat_obs` so that *healthier* patients are more
likely to be treated, and predict the direction of the bias before you run the cell.


## RCTs in the wild

Randomized experiments are everywhere once you look:

* **Medicine** — randomized clinical trials for drug approval,
* **Tech** — *A/B tests* are RCTs for product changes (button colors, ranking algorithms),
* **Economics & policy** — randomized rollouts of cash transfers, job training,
* **Agriculture** — R. A. Fisher's randomized field trials, where it all began.


When you *can* randomize, do it. But often you cannot — for cost, ethics, or because the
"treatment" already happened. That is the subject of **Lecture II**.


## ⏸ Pause the video: four headlines

For each headline, name the most likely **confounder — or reverse causation** — and state
the correct causal caveat. Write your answers down before revealing ours.

1. "People who take vitamins live longer."
2. "Students who own more books get higher test scores."
3. "Cities with more police have more crime."
4. "Coffee drinkers have higher rates of heart disease."


**Intended answers**

1. **Confounder: health-consciousness.** People who take vitamins also exercise, eat better
   and see doctors. Vitamin takers are different people; the vitamins may do nothing.
2. **Confounder: family background.** Parents' education and income drive both the
   bookshelf and the test scores. Mailing books to households is not the same intervention.
3. **Reverse causation.** High-crime cities hire more police — the arrow runs from crime to
   police. (City size is a confounder on top.) "Fewer police would mean less crime" does
   not follow.
4. **Confounder: smoking** (historically strongly associated with heavy coffee drinking)
   and the stress and sleep habits that go with it. Adjusting for smoking shrinks the
   association dramatically.


# Summary

## Summary

1. **A regression coefficient is an association, not an effect.** OLS estimates
   $P(Y \mid X)$; a causal claim is about $P(Y \mid do(X))$, and the two differ whenever
   there is confounding.
2. **A confounder is a common cause of treatment and outcome.** Adjusting for it (holding it
   fixed, or adding it to the regression) removes the spurious association — but only if you
   measured it and knew to include it.
3. **Potential outcomes $Y(1), Y(0)$ define the effect; you only ever observe one.** This is
   the fundamental problem of causal inference, so we estimate averages like the
   $\text{ATE} = \mathbb{E}[Y(1) - Y(0)]$.
4. **Naive difference in means $=$ causal effect $+$ selection bias.** Simpson's paradox is
   what selection bias looks like in a table: within-group and overall comparisons can point
   in opposite directions. Look for the confounder's two fingerprints; **direct
   standardization** gives every arm the same case mix.
5. **Randomization makes treatment independent of potential outcomes**, so the selection-bias
   term is zero, $\text{ATT} = \text{ATE}$, and the naive difference is unbiased for the ATE.


**Next:** when you cannot randomize, how do causal graphs tell you what to adjust for — and
what *not* to?


## References

- [*Causal Inference: What If*](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/) — Hernán & Robins (free PDF)
- [*The Book of Why*](http://bayes.cs.ucla.edu/WHY/) — Pearl & Mackenzie (popular intro)
- [*Causal Inference: The Mixtape*](https://mixtape.scunning.com/) — Cunningham (free online)
- [*Mastering 'Metrics*](https://www.masteringmetrics.com/) — Angrist & Pischke

## Bibliography
